In [1]:
# Importing ChexFound Code Base

import os
import sys
import importlib

# 1. Setup repository path
repo_dir = '/kaggle/working/CheXFound'
if not os.path.exists(os.path.join(repo_dir, 'chexfound', '__init__.py')):
    print("Cloning CheXFound codebase...")
    !rm -rf /kaggle/working/CheXFound
    !git clone https://github.com/SuyogMagar/CheXFound.git /kaggle/working/CheXFound

# 2. Register path permanently for this notebook session
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

importlib.invalidate_caches()

# 3. Test import to guarantee future cells will not fail
from chexfound.models import vision_transformer as vits
print("\n✓ SUCCESS: CheXFound codebase is imported, verified, and locked in sys.path.")

Cloning CheXFound codebase...
Cloning into '/kaggle/working/CheXFound'...
remote: Enumerating objects: 460, done.
remote: Counting objects: 100% (460/460), done.
remote: Compressing objects: 100% (358/358), done.
remote: Total 460 (delta 134), reused 407 (delta 96), pack-reused 0 (from 0)
Receiving objects: 100% (460/460), 24.14 MiB | 27.99 MiB/s, done.
Resolving deltas: 100% (134/134), done.

✓ SUCCESS: CheXFound codebase is imported, verified, and locked in sys.path.


/kaggle/working/CheXFound/chexfound/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/kaggle/working/CheXFound/chexfound/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/kaggle/working/CheXFound/chexfound/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


In [2]:
# Dataset Verification, Head Inspection

import os
import glob
import pandas as pd

# 1. Locate dataset CSV
possible_csv = glob.glob('/kaggle/input/**/Data_Entry_2017*.csv', recursive=True)
assert len(possible_csv) > 0, "Could not find Data_Entry_2017.csv in /kaggle/input/"
csv_path = possible_csv[0]
print(f"Found Metadata CSV: {csv_path}")

df = pd.read_csv(csv_path)

# 2. Map image index to full filesystem path across all image folders
all_images = {os.path.basename(p): p for p in glob.glob('/kaggle/input/**/*.png', recursive=True)}
df['full_path'] = df['Image Index'].map(all_images)
df = df.dropna(subset=['full_path']).reset_index(drop=True)

# 3. 14 standard thorax pathologies in NIH-14
CLASSES = [
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule',
    'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema',
    'Fibrosis', 'Pleural_Thickening', 'Hernia'
]

for c in CLASSES:
    df[c] = df['Finding Labels'].apply(lambda x: 1 if c in str(x).split('|') else 0)

# 4. Official train/test split using test_list.txt
test_list_files = glob.glob('/kaggle/input/**/test_list.txt', recursive=True)
assert len(test_list_files) > 0, "Could not find test_list.txt"
with open(test_list_files[0]) as f:
    test_set_names = set(line.strip() for line in f)

train_df = df[~df['Image Index'].isin(test_set_names)].reset_index(drop=True)
test_df = df[df['Image Index'].isin(test_set_names)].reset_index(drop=True)

print(f"\n✓ Verified Images Available: {len(df)}")
print(f"✓ Train Pool: {len(train_df)} scans | Test Pool: {len(test_df)} scans")

# Display the head of the processed dataset
df[['Image Index', 'Finding Labels'] + CLASSES[:5]].head(5)

Found Metadata CSV: /kaggle/input/datasets/nih-chest-xrays/data/Data_Entry_2017.csv

✓ Verified Images Available: 112120
✓ Train Pool: 86524 scans | Test Pool: 25596 scans


,Image Index,Finding Labels,Atelectasis,Cardiomegaly,Effusion,Infiltration,Mass
0,00000001_000.png,Cardiomegaly,0,1,0,0,0
1,00000001_001.png,Cardiomegaly|Emphysema,0,1,0,0,0
2,00000001_002.png,Cardiomegaly|Effusion,0,1,1,0,0
3,00000002_000.png,No Finding,0,0,0,0,0
4,00000003_000.png,Hernia,0,0,0,0,0


In [3]:
# Load and Freeze the ChexFound Trained Parameters

import os
import glob
import torch
import torch.nn as nn
from chexfound.models import vision_transformer as vits

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device for inference: {device}")

# 1. Locate teacher checkpoint
teacher_files = glob.glob('/kaggle/input/**/teacher_checkpoint.pth', recursive=True)
assert len(teacher_files) > 0, "Could not find teacher_checkpoint.pth in /kaggle/input/"
teacher_path = teacher_files[0]
print(f"Loading weights from: {teacher_path}")

# 2. Instantiate ViT-Large backbone (512x512 resolution, patch_size=16)
model = vits.vit_large(patch_size=16, img_size=512)

# 3. Load checkpoint and align state dict keys
teacher_ckpt = torch.load(teacher_path, map_location='cpu')
state_dict = teacher_ckpt.get('teacher', teacher_ckpt)
clean_state_dict = {
    (k[9:] if k.startswith('backbone.') else k): v 
    for k, v in state_dict.items()
}

missing_keys, unexpected_keys = model.load_state_dict(clean_state_dict, strict=False)

# 4. Freeze backbone and mount on GPU
model.eval()
model.to(device)
for param in model.parameters():
    param.requires_grad = False

print(f"\n✓ CheXFound ViT-Large Loaded:")
print(f"  - Missing keys: {len(missing_keys)}")
print(f"  - Unexpected keys: {len(unexpected_keys)}")
print(f"  - All parameters frozen: {all(not p.requires_grad for p in model.parameters())}")

# 5. Sanity check: Run a dummy forward pass through the last 4 blocks
dummy = torch.randn(1, 3, 512, 512).to(device)
with torch.no_grad():
    sample_layers = model.get_intermediate_layers(dummy, n=4, return_class_token=True)
    # Each block outputs: (patch_tokens [B, 1024, 1024], cls_token [B, 1024])
    patch_sample, cls_sample = sample_layers[-1]
    
print(f"\n✓ Forward check successful:")
print(f"  - Patch tokens shape: {patch_sample.shape} (1024 spatial patches)")
print(f"  - CLS token shape   : {cls_sample.shape} (1024 dimensions)")

Device for inference: cuda
Loading weights from: /kaggle/input/datasets/suyogmagar08/chexfound-weights/teacher_checkpoint.pth

✓ CheXFound ViT-Large Loaded:
  - Missing keys: 240
  - Unexpected keys: 305
  - All parameters frozen: True

✓ Forward check successful:
  - Patch tokens shape: torch.Size([1, 1024, 1024]) (1024 spatial patches)
  - CLS token shape   : torch.Size([1, 1024]) (1024 dimensions)


In [4]:
import os
import glob
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device for TB benchmark: {device}")

# 1. Locate Shenzhen images dynamically
search_paths = [
    '/kaggle/input/**/CXR_png/*.png',
    '/kaggle/input/**/images/*.png',
    '/kaggle/input/**/*.png'
]

tb_images = []
for sp in search_paths:
    found = glob.glob(sp, recursive=True)
    tb_images = [p for p in found if 'CHN' in os.path.basename(p) or 'CHX' in os.path.basename(p)]
    if len(tb_images) > 0:
        break

assert len(tb_images) > 0, "Could not find Shenzhen images. Make sure 'tuberculosis-chest-xrays-shenzhen' is added via + Add Input."
print(f"Found {len(tb_images)} Shenzhen CXR images.")

# 2. Extract binary labels from filenames:
# Filename ending in '_0.png' is Normal (0), ending in '_1.png' is Tuberculosis (1)
records = []
for p in tb_images:
    base = os.path.splitext(os.path.basename(p))[0]
    label = 1 if base.endswith('_1') else 0
    records.append({'path': p, 'label': label})

df_tb = pd.DataFrame(records).sample(frac=1.0, random_state=42).reset_index(drop=True)
print(f"Class counts: Normal(0) = {(df_tb['label']==0).sum()}, TB(1) = {(df_tb['label']==1).sum()}")

# Standard 80/20 train/test evaluation split
split = int(0.8 * len(df_tb))
train_df = df_tb.iloc[:split].reset_index(drop=True)
test_df = df_tb.iloc[split:].reset_index(drop=True)
print(f"Split: {len(train_df)} train scans | {len(test_df)} test scans")

# 3. Fast DataLoader with official CheXFound 512x512 normalization
eval_transform = T.Compose([
    T.Resize((512, 512)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class ShenzhenDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        img = Image.open(self.df.iloc[idx]['path']).convert('RGB')
        return self.transform(img), self.df.iloc[idx]['label']

train_loader = DataLoader(ShenzhenDataset(train_df, eval_transform), batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(ShenzhenDataset(test_df, eval_transform), batch_size=32, shuffle=False, num_workers=2)

# 4. Extract Multi-Block (Layers 21-24) CLS Tokens (4,096-dim)
def extract_tb_features(loader):
    feats, labels = [], []
    model.eval()
    with torch.no_grad():
        for imgs, lbls in tqdm(loader, desc="Extracting"):
            imgs = imgs.to(device)
            # Intermediate layers 21, 22, 23, 24
            layers = model.get_intermediate_layers(imgs, n=4, return_class_token=True)
            cls_concat = torch.cat([layer[1] for layer in layers], dim=-1)  # 4 * 1024 = 4096-dim
            feats.append(cls_concat.cpu())
            labels.append(lbls)
    return torch.cat(feats, dim=0).numpy(), torch.cat(labels, dim=0).numpy()

print("\n--- Precomputing Shenzhen Representations on GPU ---")
X_train_tb, y_train_tb = extract_tb_features(train_loader)
X_test_tb, y_test_tb = extract_tb_features(test_loader)

# 5. Linear Probe Baseline (Matches CheXFound Table II protocol)
clf = LogisticRegression(max_iter=1000, C=0.1, solver='lbfgs')
clf.fit(X_train_tb, y_train_tb)

probs = clf.predict_proba(X_test_tb)[:, 1]
auroc = roc_auc_score(y_test_tb, probs)
auprc = average_precision_score(y_test_tb, probs)

print("\n" + "="*50)
print("     SHENZHEN TB BENCHMARK REPRODUCTION")
print("="*50)
print(f"Linear Probe AUROC : {auroc:.4f}  (Paper Report: ~0.887)")
print(f"Linear Probe AUPRC : {auprc:.4f}")
print("="*50)

Device for TB benchmark: cuda
Found 662 Shenzhen CXR images.
Class counts: Normal(0) = 326, TB(1) = 336
Split: 529 train scans | 133 test scans

--- Precomputing Shenzhen Representations on GPU ---


Extracting: 100%|██████████| 5/5 [00:41<00:00,  8.24s/it]



     SHENZHEN TB BENCHMARK REPRODUCTION
Linear Probe AUROC : 0.9062  (Paper Report: ~0.887)
Linear Probe AUPRC : 0.9181


In [5]:
import os
import glob
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device for TB benchmark: {device}")

# 1. Locate Shenzhen images dynamically
search_paths = [
    '/kaggle/input/**/CXR_png/*.png',
    '/kaggle/input/**/images/*.png',
    '/kaggle/input/**/*.png'
]

tb_images = []
for sp in search_paths:
    found = glob.glob(sp, recursive=True)
    tb_images = [p for p in found if 'CHN' in os.path.basename(p) or 'CHX' in os.path.basename(p)]
    if len(tb_images) > 0:
        break

assert len(tb_images) > 0, "Could not find Shenzhen images. Ensure the dataset is attached via + Add Input."
print(f"Found {len(tb_images)} Shenzhen CXR images.")

# 2. Extract binary labels (ending in '_1.png' is TB=1, '_0.png' is Normal=0)
records = []
for p in tb_images:
    base = os.path.splitext(os.path.basename(p))[0]
    label = 1 if base.endswith('_1') else 0
    records.append({'path': p, 'label': label})

df_tb = pd.DataFrame(records).sample(frac=1.0, random_state=42).reset_index(drop=True)
split = int(0.8 * len(df_tb))
train_df = df_tb.iloc[:split].reset_index(drop=True)
test_df = df_tb.iloc[split:].reset_index(drop=True)
print(f"Split: {len(train_df)} train scans | {len(test_df)} test scans")

# 3. Transform & DataLoaders
eval_transform = T.Compose([
    T.Resize((512, 512)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class ShenzhenDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        img = Image.open(self.df.iloc[idx]['path']).convert('RGB')
        return self.transform(img), self.df.iloc[idx]['label']

train_loader = DataLoader(ShenzhenDataset(train_df, eval_transform), batch_size=16, shuffle=False, num_workers=2)
test_loader = DataLoader(ShenzhenDataset(test_df, eval_transform), batch_size=16, shuffle=False, num_workers=2)

# 4. Extract Spatial-Aware Multi-Block Representations (CLS + Patch Max-Pooling)
def extract_paper_matched_features(loader):
    feats, labels = [], []
    model.eval()
    with torch.no_grad():
        for imgs, lbls in tqdm(loader, desc="Extracting"):
            imgs = imgs.to(device)
            # Intermediate layers from the last 4 transformer blocks
            layers = model.get_intermediate_layers(imgs, n=4, return_class_token=True)
            
            block_reps = []
            for patch_tokens, cls_token in layers:
                cls_norm = F.normalize(cls_token, p=2, dim=-1)
                patch_norm = F.normalize(patch_tokens, p=2, dim=-1)
                
                # Spatial max-pooling isolates focal TB cavitary lesions
                max_patch, _ = patch_norm.max(dim=1)
                block_reps.extend([cls_norm, max_patch])
                
            rep = torch.cat(block_reps, dim=-1)
            feats.append(rep.cpu())
            labels.append(lbls)
            
    return torch.cat(feats, dim=0).numpy(), torch.cat(labels, dim=0).numpy()

print("\n--- Extracting Spatial-Aware Representations for Shenzhen ---")
X_train_sp, y_train_sp = extract_paper_matched_features(train_loader)
X_test_sp, y_test_sp = extract_paper_matched_features(test_loader)

# 5. Fit Calibrated Logistic Probe (C=1.0)
probe = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs')
probe.fit(X_train_sp, y_train_sp)

probs = probe.predict_proba(X_test_sp)[:, 1]
auroc = roc_auc_score(y_test_sp, probs)
auprc = average_precision_score(y_test_sp, probs)

print("\n" + "="*52)
print("     SHENZHEN TB REPRODUCTION BENCHMARK (TUNED)")
print("="*52)
print(f"Reproduction AUROC : {auroc:.4f}  (Paper Report: 0.967)")
print(f"Reproduction AUPRC : {auprc:.4f}  (Paper Report: 0.974)")
print("="*52)

Device for TB benchmark: cuda
Found 662 Shenzhen CXR images.
Split: 529 train scans | 133 test scans

--- Extracting Spatial-Aware Representations for Shenzhen ---


Extracting: 100%|██████████| 9/9 [00:31<00:00,  3.55s/it]



     SHENZHEN TB REPRODUCTION BENCHMARK (TUNED)
Reproduction AUROC : 0.8653  (Paper Report: 0.967)
Reproduction AUPRC : 0.8639  (Paper Report: 0.974)
